# Kronos SPUS Small v1 — Colab Free (T4)

Fine-tunes `NeoQuasar/Kronos-small` on SPUS daily CSVs. Tokenizer stays frozen.

Drive folder can be `kronos` or `Kronos` — paths are resolved case-insensitively.

Repo: https://github.com/0xboy/Kronos

## 1) GPU check

In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise SystemExit("No GPU — Runtime → Change runtime type → T4 GPU, then reconnect.")

## 2) Mount Drive + resolve Kronos/kronos folder

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

MYDRIVE = Path("/content/drive/MyDrive")

def resolve_drive_root() -> Path:
    # Prefer existing folder regardless of case (Kronos vs kronos)
    matches = [p for p in MYDRIVE.iterdir() if p.is_dir() and p.name.lower() == "kronos"]
    if matches:
        # Prefer the one that already has data/
        for p in matches:
            if (p / "data").exists() or (p / "finetuned").exists():
                return p
        return matches[0]
    root = MYDRIVE / "kronos"
    root.mkdir(parents=True, exist_ok=True)
    return root

DRIVE_ROOT = resolve_drive_root()
DATA_DIR = DRIVE_ROOT / "data" / "spus"
DATA_ZIP = DRIVE_ROOT / "data" / "spus_for_colab.zip"
FINETUNE_DIR = DRIVE_ROOT / "finetuned"
REPO_DIR = Path("/content/Kronos")

for p in (DRIVE_ROOT / "data", FINETUNE_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("MyDrive kids:", [p.name for p in MYDRIVE.iterdir() if p.is_dir()][:30])
print("DRIVE_ROOT", DRIVE_ROOT)
print("DATA_DIR  ", DATA_DIR)
print("FINETUNE  ", FINETUNE_DIR)

## 3) Clone / update repo

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/0xboy/Kronos.git"
REPO_DIR = Path("/content/Kronos")

if REPO_DIR.exists():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())
print("cwd", os.getcwd())

## 4) Install deps

In [ ]:
%pip install -q einops==0.8.1 huggingface_hub==0.33.1 safetensors==0.6.2 tqdm pyyaml matplotlib

## 5) Unpack SPUS data

File picker if zip is missing — choose `C:\\project\\kronos\\data\\spus_for_colab.zip`.

In [ ]:
import zipfile
from pathlib import Path

print("looking for zip:", DATA_ZIP)
if DATA_DIR.parent.exists():
    print("data parent:", [p.name for p in DATA_DIR.parent.iterdir()])

csv_count = len(list(DATA_DIR.glob("*.csv"))) if DATA_DIR.exists() else 0
# also accept nested extract (zip may contain a spus/ folder)
if csv_count < 50:
    nested = list(DATA_DIR.rglob("*.csv"))
    csv_count = len(nested)

print(f"existing csv count: {csv_count}")

def _unzip(zip_path: Path):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    csvs = list(DATA_DIR.glob("*.csv"))
    if len(csvs) < 50:
        # flatten one nested folder of csvs if needed
        nested_dirs = [p for p in DATA_DIR.iterdir() if p.is_dir()]
        for d in nested_dirs:
            nested_csvs = list(d.glob("*.csv"))
            if len(nested_csvs) >= 50:
                for f in nested_csvs:
                    dest = DATA_DIR / f.name
                    if not dest.exists():
                        f.replace(dest)
                break
    n = len(list(DATA_DIR.glob("*.csv")))
    print(f"unzipped -> {DATA_DIR} ({n} csv)")
    return n

if csv_count >= 50 and DATA_DIR.exists() and list(DATA_DIR.glob("*.csv")):
    print("SPUS data already present - skip unzip.")
elif DATA_ZIP.exists():
    _unzip(DATA_ZIP)
else:
    found = list(Path("/content").rglob("spus_for_colab.zip"))
    if found:
        print("found local zip:", found[0])
        DATA_ZIP.parent.mkdir(parents=True, exist_ok=True)
        DATA_ZIP.write_bytes(found[0].read_bytes())
        _unzip(DATA_ZIP)
    else:
        print("Zip not on Drive - pick spus_for_colab.zip from your PC (~7 MB).")
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise SystemExit("No file uploaded.")
        name = next(iter(uploaded))
        DATA_ZIP.parent.mkdir(parents=True, exist_ok=True)
        DATA_ZIP.write_bytes(uploaded[name])
        print("saved to Drive:", DATA_ZIP, "bytes=", DATA_ZIP.stat().st_size)
        _unzip(DATA_ZIP)

print("final csv:", len(list(DATA_DIR.glob("*.csv"))))

## 6) Write runtime config (correct Drive paths) + train

Uses your resolved `DRIVE_ROOT` so `Kronos` vs `kronos` does not matter.

In [ ]:
import os
import shutil
from pathlib import Path
import yaml

os.chdir(REPO_DIR / "finetune_csv")
print("cwd", os.getcwd())
print("DATA_DIR csv:", len(list(DATA_DIR.glob("*.csv"))))
print("DATA_DIR", DATA_DIR)

src_cfg = Path("configs/config_spus_small_v1_colab.yaml")
runtime_cfg = Path("configs/config_spus_small_v1_colab_runtime.yaml")

with open(src_cfg, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["data_path"] = str(DATA_DIR)
cfg["model_paths"]["base_path"] = str(FINETUNE_DIR)

with open(runtime_cfg, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print("runtime config:", runtime_cfg.resolve())
print("data_path:", cfg["data"]["data_path"])
print("base_path:", cfg["model_paths"]["base_path"])

# Use shell so training logs stream live in Colab
!python train_sequential.py --config configs/config_spus_small_v1_colab_runtime.yaml --skip-tokenizer

## 7) Checkpoint on Drive

In [ ]:
best = FINETUNE_DIR / "spus_small_v1" / "basemodel" / "best_model"
print("best_model dir:", best)
if best.exists():
    for p in sorted(best.iterdir()):
        if p.is_file():
            mb = p.stat().st_size / 1e6
            print(f"  {p.name:40s} {mb:8.2f} MB")
        else:
            print(f"  {p.name}/")
else:
    print("Not found yet — training may still be running or failed.")

print("\nLocal paper alias already points at spus-small-v1.")
print("After download, put files under:")
print("  finetune_csv/finetuned/spus_small_v1/basemodel/best_model/")

### Tips
- Session idle disconnects kill `/content` but **Drive checkpoints survive**.
- OOM → set `batch_size: 4` in runtime config cell and re-run.
- If train fails, scroll up in the same cell for the real Python traceback (now streamed).